# Time Series Analysis for Quant Finance

A rigorous treatment of time series theory and methods, covering the essential material for quantitative researcher, quantitative trader, and SWE interviews at top firms.

**Prerequisites:** Probability theory, linear algebra, basic statistics, stochastic processes.

**Topics covered:**
1. Stationarity and Unit Root Tests
2. Autoregressive (AR) Models
3. Moving Average (MA) Models
4. ARMA / ARIMA Models and Box-Jenkins Methodology
5. GARCH Family Models
6. Cointegration and Error Correction
7. Kalman Filter and State-Space Models
8. Spectral Analysis
9. Finance Applications and Interview Problems

---
## 1. Stationarity

Stationarity is the foundational concept of time series analysis. Nearly every classical model assumes some form of stationarity, and understanding when and why this assumption fails is critical in finance.

### 1.1 Strict (Strong) Stationarity

A stochastic process $\{X_t\}$ is **strictly stationary** if the joint distribution of any finite collection of observations is invariant under time shifts:

$$F_{X_{t_1}, X_{t_2}, \ldots, X_{t_k}}(x_1, x_2, \ldots, x_k) = F_{X_{t_1+h}, X_{t_2+h}, \ldots, X_{t_k+h}}(x_1, x_2, \ldots, x_k)$$

for all $k \geq 1$, all times $t_1, \ldots, t_k$, all shifts $h$, and all $(x_1, \ldots, x_k) \in \mathbb{R}^k$.

This is an extremely strong condition. An i.i.d. sequence is strictly stationary. A Gaussian process that is weakly stationary is also strictly stationary (since the Gaussian distribution is fully determined by its first two moments).

### 1.2 Weak (Wide-Sense / Covariance) Stationarity

A process $\{X_t\}$ is **weakly stationary** (or covariance stationary) if:

1. $\mathbb{E}[X_t] = \mu$ for all $t$ (constant mean),
2. $\text{Var}(X_t) = \sigma^2 < \infty$ for all $t$ (constant, finite variance),
3. $\text{Cov}(X_t, X_{t+h}) = \gamma(h)$ depends only on the lag $h$, not on $t$.

The function $\gamma(h) = \text{Cov}(X_t, X_{t+h})$ is the **autocovariance function** (ACVF). The **autocorrelation function** (ACF) is:

$$\rho(h) = \frac{\gamma(h)}{\gamma(0)}$$

> **Key properties of the ACVF:**
> - $\gamma(0) = \text{Var}(X_t) \geq 0$
> - $\gamma(h) = \gamma(-h)$ (symmetry)
> - $|\gamma(h)| \leq \gamma(0)$ (Cauchy-Schwarz)
> - $\gamma(\cdot)$ is positive semi-definite: $\sum_{i=1}^{n}\sum_{j=1}^{n} a_i a_j \gamma(t_i - t_j) \geq 0$ for all $\{a_i\}$

### 1.3 Unit Root Tests

In practice, we need formal tests to determine if a series is stationary.

**Augmented Dickey-Fuller (ADF) Test.** Consider the model:

$$\Delta X_t = \alpha + \beta t + \gamma X_{t-1} + \sum_{i=1}^{p} \delta_i \Delta X_{t-i} + \varepsilon_t$$

- $H_0$: $\gamma = 0$ (unit root exists, series is non-stationary)
- $H_1$: $\gamma < 0$ (series is stationary)

The test statistic does **not** follow a standard $t$-distribution; it follows the **Dickey-Fuller distribution**. Critical values are tabulated (or computed via simulation). Reject $H_0$ when the test statistic is sufficiently negative.

**KPSS Test.** The Kwiatkowski-Phillips-Schmidt-Shin test reverses the hypotheses:

- $H_0$: The series is (trend-)stationary
- $H_1$: The series has a unit root

The model decomposes $X_t = \beta t + r_t + \varepsilon_t$, where $r_t = r_{t-1} + u_t$ is a random walk. Under $H_0$, $\text{Var}(u_t) = 0$.

**Phillips-Perron Test.** A non-parametric modification of the ADF test that corrects for serial correlation and heteroskedasticity using the Newey-West estimator, rather than adding lagged differences.

> 💡 **Interview Tip:** Using ADF and KPSS together is a standard diagnostic strategy. If ADF rejects (stationary) and KPSS does not reject (stationary), conclude stationarity. If ADF does not reject (unit root) and KPSS rejects (not stationary), conclude unit root. Conflicting results suggest fractional integration or structural breaks.

### 1.4 Achieving Stationarity: Differencing and Detrending

**Differencing.** If $X_t$ is integrated of order $d$, denoted $X_t \sim I(d)$, then $(1 - L)^d X_t$ is stationary, where $L$ is the **lag operator** ($LX_t = X_{t-1}$).

- First differencing: $\Delta X_t = X_t - X_{t-1} = (1-L)X_t$
- Second differencing: $\Delta^2 X_t = \Delta X_t - \Delta X_{t-1}$

Financial log-prices are typically $I(1)$, so log-returns $r_t = \ln(P_t/P_{t-1}) = \Delta \ln P_t$ are (approximately) stationary.

**Detrending.** If the non-stationarity is due to a deterministic trend $X_t = f(t) + Y_t$ where $Y_t$ is stationary, we can regress $X_t$ on $f(t)$ and analyze the residuals. Common choices: linear trend $f(t) = \alpha + \beta t$, polynomial trend, or the Hodrick-Prescott filter.

> **Important distinction:** Differencing is appropriate for stochastic trends (unit roots). Detrending is appropriate for deterministic trends. Applying the wrong transformation can distort inference.

---
## 2. Autoregressive (AR) Models

### 2.1 AR(1) Model

The first-order autoregressive model is:

$$X_t = c + \phi X_{t-1} + \varepsilon_t, \qquad \varepsilon_t \sim \text{WN}(0, \sigma^2)$$

where $\text{WN}(0,\sigma^2)$ denotes white noise.

**Stationarity condition:** $|\phi| < 1$.

When $|\phi| < 1$, repeated back-substitution gives the causal (infinite MA) representation:

$$X_t = \frac{c}{1-\phi} + \sum_{j=0}^{\infty} \phi^j \varepsilon_{t-j}$$

**Moments under stationarity ($|\phi|<1$):**

\begin{align*}
\mathbb{E}[X_t] &= \mu = \frac{c}{1-\phi} \\
\text{Var}(X_t) &= \gamma(0) = \frac{\sigma^2}{1-\phi^2} \\
\gamma(h) &= \phi^{|h|} \cdot \frac{\sigma^2}{1-\phi^2} \\
\rho(h) &= \phi^{|h|}
\end{align*}

Key observations:
- The ACF decays **geometrically** (exponentially) with lag.
- If $\phi > 0$: all autocorrelations are positive (persistence).
- If $\phi < 0$: autocorrelations alternate in sign (mean-reversion at each step).
- The **partial autocorrelation function** (PACF) has a single spike at lag 1, then cuts off to zero.

**Half-life of mean reversion:** The time for a shock to decay to half its initial magnitude is:

$$h_{1/2} = \frac{\ln(1/2)}{\ln|\phi|} = -\frac{\ln 2}{\ln|\phi|}$$

### 2.2 AR(p) Model and the Characteristic Equation

The general $p$-th order AR model is:

$$X_t = c + \phi_1 X_{t-1} + \phi_2 X_{t-2} + \cdots + \phi_p X_{t-p} + \varepsilon_t$$

Using the lag operator $L$:

$$\Phi(L) X_t = c + \varepsilon_t, \qquad \Phi(L) = 1 - \phi_1 L - \phi_2 L^2 - \cdots - \phi_p L^p$$

**Stationarity condition:** The AR(p) process is stationary if and only if all roots of the **characteristic polynomial**

$$\Phi(z) = 1 - \phi_1 z - \phi_2 z^2 - \cdots - \phi_p z^p = 0$$

lie **outside** the unit circle in the complex plane (i.e., $|z_i| > 1$ for all roots $z_i$).

Equivalently, all eigenvalues of the companion matrix

$$\mathbf{F} = \begin{pmatrix} \phi_1 & \phi_2 & \cdots & \phi_{p-1} & \phi_p \\ 1 & 0 & \cdots & 0 & 0 \\ 0 & 1 & \cdots & 0 & 0 \\ \vdots & & \ddots & & \vdots \\ 0 & 0 & \cdots & 1 & 0 \end{pmatrix}$$

must have modulus strictly less than 1.

**Example: AR(2).** The model $X_t = \phi_1 X_{t-1} + \phi_2 X_{t-2} + \varepsilon_t$ is stationary iff:

\begin{align*}
\phi_1 + \phi_2 &< 1 \\
\phi_2 - \phi_1 &< 1 \\
|\phi_2| &< 1
\end{align*}

These define a triangular region in the $(\phi_1, \phi_2)$ plane.

### 2.3 Yule-Walker Equations

The Yule-Walker equations link the AR parameters to the autocovariances. Multiplying the AR(p) equation by $X_{t-h}$ and taking expectations:

$$\gamma(h) = \phi_1 \gamma(h-1) + \phi_2 \gamma(h-2) + \cdots + \phi_p \gamma(h-p) \qquad \text{for } h \geq 1$$

In matrix form for $h = 1, \ldots, p$:

$$\begin{pmatrix} \gamma(0) & \gamma(1) & \cdots & \gamma(p-1) \\ \gamma(1) & \gamma(0) & \cdots & \gamma(p-2) \\ \vdots & \vdots & \ddots & \vdots \\ \gamma(p-1) & \gamma(p-2) & \cdots & \gamma(0) \end{pmatrix} \begin{pmatrix} \phi_1 \\ \phi_2 \\ \vdots \\ \phi_p \end{pmatrix} = \begin{pmatrix} \gamma(1) \\ \gamma(2) \\ \vdots \\ \gamma(p) \end{pmatrix}$$

Or compactly: $\mathbf{\Gamma}_p \boldsymbol{\phi} = \boldsymbol{\gamma}_p$, where $\mathbf{\Gamma}_p$ is the Toeplitz autocovariance matrix.

The Yule-Walker estimator is $\hat{\boldsymbol{\phi}} = \hat{\mathbf{\Gamma}}_p^{-1} \hat{\boldsymbol{\gamma}}_p$, using sample autocovariances.

The noise variance is recovered from:

$$\sigma^2 = \gamma(0) - \phi_1 \gamma(1) - \phi_2 \gamma(2) - \cdots - \phi_p \gamma(p)$$

> 💡 **Interview Tip:** Interviewers often ask you to derive properties of AR(1) from scratch. Be able to compute $\mathbb{E}[X_t]$, $\text{Var}(X_t)$, and $\text{Cov}(X_t, X_{t-h})$ by direct calculation (substitute and take expectations). Also know the connection between the Yule-Walker equations and the method of moments.

### Worked Problem 1: AR(1) Variance and Autocorrelation

> **Problem.** Let $X_t = 0.6 X_{t-1} + \varepsilon_t$ where $\varepsilon_t \sim \text{WN}(0, 4)$. Find the unconditional variance of $X_t$, the autocorrelation at lag 3, and the half-life of shocks.

**Solution.**

Here $\phi = 0.6$, $\sigma^2 = 4$, and $c = 0$ so $\mu = 0$.

*Variance:*

$$\gamma(0) = \frac{\sigma^2}{1 - \phi^2} = \frac{4}{1 - 0.36} = \frac{4}{0.64} = 6.25$$

*Autocorrelation at lag 3:*

$$\rho(3) = \phi^3 = 0.6^3 = 0.216$$

*Half-life:*

$$h_{1/2} = -\frac{\ln 2}{\ln 0.6} = -\frac{0.6931}{-0.5108} \approx 1.357 \text{ periods}$$

So shocks decay to half their magnitude in approximately 1.36 time steps.

---
## 3. Moving Average (MA) Models

### 3.1 MA(1) Model

$$X_t = \mu + \varepsilon_t + \theta \varepsilon_{t-1}, \qquad \varepsilon_t \sim \text{WN}(0, \sigma^2)$$

**Key fact:** An MA(q) process is **always** stationary for any finite parameter values, since it is a finite linear combination of white noise terms.

**Moments of MA(1):**

\begin{align*}
\mathbb{E}[X_t] &= \mu \\
\gamma(0) &= (1 + \theta^2)\sigma^2 \\
\gamma(1) &= \theta \sigma^2 \\
\gamma(h) &= 0 \quad \text{for } |h| \geq 2
\end{align*}

So the ACF has exactly one non-zero value at lag 1:

$$\rho(1) = \frac{\theta}{1 + \theta^2}, \qquad \rho(h) = 0 \text{ for } |h| \geq 2$$

Notice that $|\rho(1)| \leq 1/2$ (with equality when $\theta = \pm 1$). This is because $\frac{\theta}{1+\theta^2}$ achieves its extremum at $\theta = \pm 1$ by AM-GM.

### 3.2 MA(q) Model

$$X_t = \mu + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \theta_2 \varepsilon_{t-2} + \cdots + \theta_q \varepsilon_{t-q} = \mu + \Theta(L)\varepsilon_t$$

where $\Theta(L) = 1 + \theta_1 L + \cdots + \theta_q L^q$.

**Autocovariance structure:**

$$\gamma(h) = \begin{cases} \sigma^2 \sum_{j=0}^{q-|h|} \theta_j \theta_{j+|h|} & \text{if } |h| \leq q \\ 0 & \text{if } |h| > q \end{cases}$$

where $\theta_0 = 1$. The ACF **cuts off** after lag $q$ (this is the signature of an MA model).

### 3.3 Invertibility

An MA(q) model is **invertible** if all roots of $\Theta(z) = 0$ lie outside the unit circle.

For MA(1): invertibility requires $|\theta| < 1$.

**Why invertibility matters:** If an MA process is invertible, we can express it as an AR($\infty$) process:

$$\varepsilon_t = \Theta(L)^{-1}(X_t - \mu) = \sum_{j=0}^{\infty} \pi_j (X_t - \mu)_{t-j}$$

This means past shocks $\varepsilon_t$ can be recovered from past observations, which is essential for forecasting and estimation.

**Non-uniqueness:** The MA(1) models with parameters $\theta$ and $1/\theta$ produce the same autocovariance structure (since $\rho(1) = \theta/(1+\theta^2) = (1/\theta)/(1+1/\theta^2)$). Invertibility resolves this ambiguity by selecting the unique representation with $|\theta| < 1$.

### 3.4 Wold's Decomposition Theorem

**Theorem (Wold, 1938).** Every weakly stationary process $\{X_t\}$ has the representation:

$$X_t = \sum_{j=0}^{\infty} \psi_j \varepsilon_{t-j} + V_t$$

where:
- $\varepsilon_t \sim \text{WN}(0, \sigma^2)$ with $\psi_0 = 1$ and $\sum_{j=0}^{\infty} \psi_j^2 < \infty$
- $V_t$ is a deterministic process (perfectly predictable from its past)
- The two components are uncorrelated

This theorem provides the theoretical foundation for ARMA modeling: any purely non-deterministic stationary process is an MA($\infty$), which can be approximated by a finite ARMA model.

### Worked Problem 2: MA(1) Identification

> **Problem.** You estimate the sample ACF of a stationary series and find $\hat{\rho}(1) = 0.4$, $\hat{\rho}(h) \approx 0$ for $h \geq 2$. (a) What model is suggested? (b) Estimate the MA parameter. (c) Is the model invertible?

**Solution.**

**(a)** The ACF cuts off after lag 1, which is the signature of an **MA(1)** model.

**(b)** We need $\rho(1) = \theta / (1 + \theta^2) = 0.4$. This gives:

$$\theta^2 - 2.5\theta + 1 = 0$$

By the quadratic formula:

$$\theta = \frac{2.5 \pm \sqrt{6.25 - 4}}{2} = \frac{2.5 \pm 1.5}{2}$$

So $\theta = 2$ or $\theta = 0.5$.

**(c)** We choose the **invertible** solution $\theta = 0.5$ (since $|0.5| < 1$). The model $\theta = 2$ generates the same autocovariance structure but is not invertible.

---
## 4. ARMA and ARIMA Models

### 4.1 ARMA(p,q) Model

The ARMA(p,q) model combines AR and MA components:

$$X_t = c + \phi_1 X_{t-1} + \cdots + \phi_p X_{t-p} + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \cdots + \theta_q \varepsilon_{t-q}$$

In operator notation:

$$\Phi(L) X_t = c + \Theta(L) \varepsilon_t$$

- **Stationarity** requires all roots of $\Phi(z) = 0$ to lie outside the unit circle.
- **Invertibility** requires all roots of $\Theta(z) = 0$ to lie outside the unit circle.
- The model is **causal** (can be written as a one-sided MA($\infty$)) iff the stationarity condition holds.

**ACF behavior:** For an ARMA(p,q), the ACF behaves like that of an AR(p) process for lags $h > q$. It decays exponentially/sinusoidally but may have irregular behavior for the first $q$ lags.

**PACF behavior:** Behaves like that of an MA(q) process for lags $h > p$. Cuts off (approximately) after lag $p$.

| Model | ACF | PACF |
|-------|-----|------|
| AR(p) | Tails off (exponential/oscillating decay) | Cuts off after lag $p$ |
| MA(q) | Cuts off after lag $q$ | Tails off |
| ARMA(p,q) | Tails off (AR-like after lag $q$) | Tails off (MA-like after lag $p$) |

### 4.2 ARIMA(p,d,q) and Box-Jenkins Methodology

If $X_t$ requires $d$ differences to become stationary, we model:

$$\Phi(L)(1-L)^d X_t = c + \Theta(L)\varepsilon_t$$

This is the **ARIMA(p,d,q)** model. In finance, $d = 1$ is common (prices are $I(1)$, returns are $I(0)$).

**Box-Jenkins Methodology (three stages):**

1. **Identification:** Determine $d$ (via unit root tests), then examine the ACF and PACF of the differenced series to guess $p$ and $q$.

2. **Estimation:** Estimate the parameters $\{\phi_i, \theta_j, \sigma^2\}$ by maximum likelihood (or conditional least squares). The likelihood for a Gaussian ARMA is:

$$\ell(\boldsymbol{\phi}, \boldsymbol{\theta}, \sigma^2) = -\frac{n}{2}\ln(2\pi) - \frac{1}{2}\ln|\boldsymbol{\Gamma}_n| - \frac{1}{2}\mathbf{X}^\top \boldsymbol{\Gamma}_n^{-1} \mathbf{X}$$

3. **Diagnostic checking:** Examine residuals $\hat{\varepsilon}_t$. They should resemble white noise (no significant ACF values, pass the Ljung-Box test).

**Ljung-Box test statistic:**

$$Q(m) = n(n+2) \sum_{h=1}^{m} \frac{\hat{\rho}^2(h)}{n-h} \sim \chi^2(m - p - q) \quad \text{under } H_0$$

Reject if $Q(m)$ is large (residuals are not white noise).

### 4.3 Model Selection: AIC and BIC

When comparing candidate ARMA(p,q) models, use information criteria:

\begin{align*}
\text{AIC} &= -2\ell + 2k \\
\text{BIC} &= -2\ell + k \ln(n)
\end{align*}

where $\ell$ is the maximized log-likelihood, $k = p + q + 1$ is the number of estimated parameters (including $\sigma^2$), and $n$ is the sample size.

- **AIC** tends to select larger models (better for forecasting).
- **BIC** penalizes complexity more and is **consistent** (selects the true model as $n \to \infty$).
- **AICc** (corrected AIC) adds a finite-sample correction: $\text{AICc} = \text{AIC} + \frac{2k(k+1)}{n-k-1}$.

Select the model with the **smallest** information criterion.

> 💡 **Interview Tip:** A common interview question asks about the trade-off between AIC and BIC. Know that AIC is asymptotically efficient (minimizes prediction error) but inconsistent, while BIC is consistent but may underfit in small samples. For trading strategy development, out-of-sample cross-validation often trumps both.

### 4.4 Forecasting with ARMA

The minimum mean squared error (MMSE) forecast of $X_{t+h}$ given information up to time $t$ is the conditional expectation:

$$\hat{X}_{t+h|t} = \mathbb{E}[X_{t+h} \mid X_t, X_{t-1}, \ldots]$$

For an ARMA(1,1): $X_{t+1} = c + \phi X_t + \varepsilon_{t+1} + \theta \varepsilon_t$, so:

$$\hat{X}_{t+1|t} = c + \phi X_t + \theta \hat{\varepsilon}_t$$

since $\mathbb{E}[\varepsilon_{t+1} | \mathcal{F}_t] = 0$ and $\varepsilon_t$ is known (or estimated) at time $t$.

The $h$-step forecast error variance for AR(1) is:

$$\text{Var}(X_{t+h} - \hat{X}_{t+h|t}) = \sigma^2 \sum_{j=0}^{h-1} \phi^{2j} = \sigma^2 \frac{1 - \phi^{2h}}{1 - \phi^2}$$

As $h \to \infty$, this converges to $\gamma(0) = \sigma^2/(1-\phi^2)$, the unconditional variance. The forecast converges to the unconditional mean $\mu = c/(1-\phi)$.

### Worked Problem 3: ARMA Forecasting

> **Problem.** Consider the ARMA(1,1) model $X_t = 0.8 X_{t-1} + \varepsilon_t - 0.5 \varepsilon_{t-1}$ with $\sigma^2 = 1$. Suppose $X_{100} = 2$ and $\hat{\varepsilon}_{100} = 0.3$. Compute the 1-step and 2-step ahead forecasts.

**Solution.**

*1-step forecast:*

$$\hat{X}_{101|100} = 0.8 \cdot X_{100} + \mathbb{E}[\varepsilon_{101}|\mathcal{F}_{100}] - 0.5 \cdot \varepsilon_{100} = 0.8(2) + 0 - 0.5(0.3) = 1.6 - 0.15 = 1.45$$

*2-step forecast:*

$$\hat{X}_{102|100} = 0.8 \cdot \hat{X}_{101|100} + \mathbb{E}[\varepsilon_{102}|\mathcal{F}_{100}] - 0.5 \cdot \mathbb{E}[\varepsilon_{101}|\mathcal{F}_{100}]$$

Both $\varepsilon_{101}$ and $\varepsilon_{102}$ are future innovations with conditional expectation zero:

$$\hat{X}_{102|100} = 0.8(1.45) + 0 - 0 = 1.16$$

Note how the MA component only affects the 1-step forecast. For $h \geq 2$, the forecast behaves like a pure AR(1) forecast applied iteratively.

---
## 5. GARCH Family Models

Financial returns exhibit **volatility clustering** (large moves tend to be followed by large moves) and heavy tails. The GARCH family captures these features by modeling conditional heteroskedasticity.

### 5.1 ARCH(1) Model

Engle (1982) proposed the **Autoregressive Conditional Heteroskedasticity** model:

\begin{align*}
r_t &= \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t, \quad z_t \sim \text{i.i.d. } N(0,1) \\
\sigma_t^2 &= \omega + \alpha \varepsilon_{t-1}^2
\end{align*}

with $\omega > 0$, $\alpha \geq 0$.

**Properties:**
- $\mathbb{E}[\varepsilon_t | \mathcal{F}_{t-1}] = 0$ (conditional mean zero)
- $\text{Var}(\varepsilon_t | \mathcal{F}_{t-1}) = \sigma_t^2 = \omega + \alpha \varepsilon_{t-1}^2$ (conditional variance depends on past)
- Unconditional variance: $\text{Var}(\varepsilon_t) = \frac{\omega}{1-\alpha}$ (requires $\alpha < 1$)
- The kurtosis of $\varepsilon_t$ exceeds 3 (heavier tails than Gaussian), which is:

$$\kappa = \frac{3(1-\alpha^2)}{1-3\alpha^2}$$

This requires $3\alpha^2 < 1$, i.e., $\alpha < 1/\sqrt{3} \approx 0.577$ for finite fourth moment.

### 5.2 GARCH(1,1) Model

Bollerslev (1986) generalized ARCH to include lagged conditional variance:

\begin{align*}
\varepsilon_t &= \sigma_t z_t, \qquad z_t \sim \text{i.i.d.}(0,1) \\
\sigma_t^2 &= \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2
\end{align*}

with $\omega > 0$, $\alpha \geq 0$, $\beta \geq 0$.

**Stationarity:** Covariance stationarity requires $\alpha + \beta < 1$.

**Unconditional variance:**

$$\bar{\sigma}^2 = \mathbb{E}[\sigma_t^2] = \frac{\omega}{1 - \alpha - \beta}$$

**Variance targeting reparameterization:** Substitute $\omega = \bar{\sigma}^2(1 - \alpha - \beta)$:

$$\sigma_t^2 = \bar{\sigma}^2(1 - \alpha - \beta) + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

**Persistence:** The quantity $\alpha + \beta$ measures volatility persistence. Typical estimates for daily equity returns give $\alpha + \beta \approx 0.98$--$0.995$, indicating very persistent volatility.

**Half-life of volatility shocks:**

$$h_{1/2} = \frac{\ln(1/2)}{\ln(\alpha + \beta)}$$

**IGARCH:** When $\alpha + \beta = 1$ (integrated GARCH), shocks to variance persist forever. The unconditional variance is infinite.

### 5.3 Asymmetric GARCH Models

Empirically, negative returns increase future volatility more than positive returns of the same magnitude (the **leverage effect**). Standard GARCH does not capture this since $\varepsilon_{t-1}^2$ is symmetric in $\varepsilon_{t-1}$.

**GJR-GARCH (Glosten-Jagannathan-Runkle):**

$$\sigma_t^2 = \omega + (\alpha + \gamma \mathbb{1}_{\{\varepsilon_{t-1}<0\}})\varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

The parameter $\gamma > 0$ adds extra impact from negative shocks.

**EGARCH (Exponential GARCH, Nelson 1991):**

$$\ln \sigma_t^2 = \omega + \alpha \left(|z_{t-1}| - \mathbb{E}|z_{t-1}|\right) + \gamma z_{t-1} + \beta \ln \sigma_{t-1}^2$$

Advantages of EGARCH:
- No positivity constraints needed (we model $\ln \sigma_t^2$, which is automatically positive when exponentiated).
- Asymmetry is captured by $\gamma$: if $\gamma < 0$, negative shocks ($z_{t-1} < 0$) increase volatility more.

**News Impact Curve (NIC):** The NIC plots $\sigma_t^2$ as a function of $\varepsilon_{t-1}$ (or $z_{t-1}$), holding $\sigma_{t-1}^2$ fixed.

- GARCH(1,1): symmetric parabola centered at $\varepsilon_{t-1} = 0$.
- GJR-GARCH: asymmetric, steeper for $\varepsilon_{t-1} < 0$.
- EGARCH: asymmetric and exponential in shape.

### Worked Problem 4: GARCH Volatility Forecast

> **Problem.** A GARCH(1,1) model for daily returns has $\omega = 0.000002$, $\alpha = 0.08$, $\beta = 0.91$. Today's return shock is $\varepsilon_t = -0.02$ and the current conditional variance is $\sigma_t^2 = 0.0003$. (a) Compute tomorrow's conditional variance. (b) What is the long-run (unconditional) variance? (c) What is the volatility half-life? (d) Is the model covariance stationary?

**Solution.**

**(a)** Tomorrow's conditional variance:

\begin{align*}
\sigma_{t+1}^2 &= \omega + \alpha \varepsilon_t^2 + \beta \sigma_t^2 \\
&= 0.000002 + 0.08 \times (-0.02)^2 + 0.91 \times 0.0003 \\
&= 0.000002 + 0.08 \times 0.0004 + 0.000273 \\
&= 0.000002 + 0.000032 + 0.000273 \\
&= 0.000307
\end{align*}

So the conditional volatility is $\sigma_{t+1} = \sqrt{0.000307} \approx 1.75\%$ daily.

**(b)** Long-run variance:

$$\bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta} = \frac{0.000002}{1 - 0.08 - 0.91} = \frac{0.000002}{0.01} = 0.0002$$

Long-run annualized volatility: $\sqrt{252 \times 0.0002} \approx 22.4\%$.

**(c)** Persistence is $\alpha + \beta = 0.99$. Half-life:

$$h_{1/2} = \frac{-\ln 2}{\ln 0.99} = \frac{0.6931}{0.01005} \approx 69 \text{ days}$$

**(d)** Since $\alpha + \beta = 0.99 < 1$, yes, the model is covariance stationary.

---
## 6. Cointegration and Error Correction Models

Cointegration is the backbone of statistical arbitrage (pairs trading, basket trading). Two $I(1)$ series may drift apart individually but maintain a long-run equilibrium.

### 6.1 Definition of Cointegration

**Definition.** Two $I(1)$ series $X_t$ and $Y_t$ are **cointegrated** of order $CI(1,1)$ if there exists a constant $\beta$ such that:

$$Z_t = Y_t - \beta X_t \sim I(0)$$

The vector $(1, -\beta)$ is called the **cointegrating vector**, and $Z_t$ is the **cointegrating residual** (or spread).

More generally, a vector $\mathbf{X}_t = (X_{1t}, \ldots, X_{nt})^\top$ of $I(1)$ processes is cointegrated if there exists a non-zero vector $\boldsymbol{\beta}$ such that $\boldsymbol{\beta}^\top \mathbf{X}_t \sim I(0)$.

**Key insight for finance:** Individual stock prices are $I(1)$ (non-stationary), but a carefully chosen linear combination of prices may be $I(0)$ (mean-reverting). This is the basis for pairs trading.

### 6.2 Engle-Granger Two-Step Procedure

**Step 1:** Run the cointegrating regression:

$$Y_t = \alpha + \beta X_t + Z_t$$

via OLS. Despite $X_t$ and $Y_t$ being $I(1)$, the OLS estimator $\hat{\beta}$ is **superconsistent** (converges at rate $T$ instead of $\sqrt{T}$) when cointegration exists.

**Step 2:** Test the residuals $\hat{Z}_t = Y_t - \hat{\alpha} - \hat{\beta}X_t$ for a unit root using the ADF test. If $\hat{Z}_t$ is stationary, conclude cointegration. Critical values are different from the standard ADF (use the Engle-Granger / Phillips-Ouliaris tables).

**Caution:** The Engle-Granger method assumes a single cointegrating relationship and may suffer from small-sample bias in the cointegrating regression.

### 6.3 Johansen Test

For multivariate systems with potentially multiple cointegrating vectors, the **Johansen test** uses the Vector Error Correction Model (VECM):

$$\Delta \mathbf{X}_t = \boldsymbol{\Pi} \mathbf{X}_{t-1} + \sum_{i=1}^{k-1} \boldsymbol{\Gamma}_i \Delta \mathbf{X}_{t-i} + \boldsymbol{\varepsilon}_t$$

where $\boldsymbol{\Pi} = \boldsymbol{\alpha}\boldsymbol{\beta}^\top$ with:
- $\boldsymbol{\beta}$: matrix of cointegrating vectors ($n \times r$)
- $\boldsymbol{\alpha}$: matrix of adjustment (loading) coefficients ($n \times r$)
- $r = \text{rank}(\boldsymbol{\Pi})$: number of cointegrating relationships

The Johansen test determines $r$ using two test statistics:

- **Trace test:** $\lambda_{\text{trace}}(r_0) = -T \sum_{i=r_0+1}^{n} \ln(1 - \hat{\lambda}_i)$, testing $H_0: r \leq r_0$ vs $H_1: r > r_0$.
- **Maximum eigenvalue test:** $\lambda_{\max}(r_0) = -T \ln(1 - \hat{\lambda}_{r_0+1})$, testing $H_0: r = r_0$ vs $H_1: r = r_0 + 1$.

Here $\hat{\lambda}_1 > \hat{\lambda}_2 > \cdots > \hat{\lambda}_n$ are the eigenvalues from the reduced-rank regression.

### 6.4 Error Correction Models (ECM)

The **Granger representation theorem** states that if $X_t$ and $Y_t$ are cointegrated, there exists an error correction representation:

\begin{align*}
\Delta Y_t &= \alpha_Y (Y_{t-1} - \beta X_{t-1}) + \text{lagged } \Delta Y, \Delta X + \varepsilon_{Y,t} \\
\Delta X_t &= \alpha_X (Y_{t-1} - \beta X_{t-1}) + \text{lagged } \Delta Y, \Delta X + \varepsilon_{X,t}
\end{align*}

The term $(Y_{t-1} - \beta X_{t-1})$ is the **error correction term** (deviation from equilibrium). The coefficients $\alpha_Y$ and $\alpha_X$ are the **speed of adjustment** parameters.

For the system to be stable and mean-revert, we need $\alpha_Y < 0$ (when $Y$ is above equilibrium, it adjusts downward) and/or $\alpha_X > 0$.

> 💡 **Interview Tip:** In a pairs trading context, the spread $Z_t = Y_t - \beta X_t$ should be stationary and mean-reverting. You enter a trade when $Z_t$ deviates significantly from its mean (e.g., beyond 2 standard deviations) and exit when it reverts. The speed of adjustment $\alpha$ determines how quickly the spread reverts and thus the expected holding period. Faster mean-reversion ($|\alpha|$ larger) is preferred.

### 6.5 Pairs Trading Application

**Practical implementation:**

1. **Pair selection:** Screen for cointegrated pairs using Engle-Granger (or Johansen) among stocks in the same sector.

2. **Spread construction:** $Z_t = Y_t - \hat{\beta} X_t$ where $\hat{\beta}$ is the cointegrating coefficient (the hedge ratio).

3. **Signal generation:** Standardize the spread:

$$s_t = \frac{Z_t - \bar{Z}}{\hat{\sigma}_Z}$$

   - Enter long spread (buy $Y$, sell $\beta$ units of $X$) when $s_t < -k$ (e.g., $k = 2$)
   - Enter short spread (sell $Y$, buy $\beta$ units of $X$) when $s_t > k$
   - Exit when $|s_t| < \delta$ (e.g., $\delta = 0.5$) or after a stop-loss

4. **Risk management:** Monitor for cointegration breakdown (rolling ADF tests), regime changes, and structural breaks.

**Key risks:** Cointegration may break down (pair diverges permanently), estimation error in $\beta$, and transaction costs may erode profits on small mean-reversions.

### Worked Problem 5: Cointegration and Pairs Trading

> **Problem.** You run a cointegrating regression of stock $Y$ on stock $X$ and obtain $Y_t = 0.5 + 1.2 X_t + Z_t$. The ADF test on $\hat{Z}_t$ rejects at the 5% level. The estimated ECM gives $\Delta Z_t = -0.15 Z_{t-1} + \varepsilon_t$ with $\hat{\sigma}_Z = 0.8$. (a) What is the hedge ratio? (b) What is the half-life of the spread? (c) If $Z_{100} = 2.4$, describe the trading signal.

**Solution.**

**(a)** The hedge ratio is $\hat{\beta} = 1.2$. For every 1 share of $Y$, hedge with 1.2 shares of $X$.

**(b)** The spread follows $\Delta Z_t = -0.15 Z_{t-1} + \varepsilon_t$, which can be rewritten as $Z_t = (1-0.15)Z_{t-1} + \varepsilon_t = 0.85 Z_{t-1} + \varepsilon_t$.

This is an AR(1) with coefficient $\phi = 0.85$. The half-life is:

$$h_{1/2} = -\frac{\ln 2}{\ln 0.85} = \frac{0.6931}{0.1625} \approx 4.3 \text{ days}$$

**(c)** The standardized spread is:

$$s_{100} = \frac{Z_{100} - 0}{0.8} = \frac{2.4}{0.8} = 3.0$$

(The equilibrium mean of $Z_t$ is 0 for a demeaned spread, or close to 0 if the intercept is small.)

Since $s_{100} = 3.0 > 2$, the spread is significantly above its mean. The signal is to **short the spread**: sell $Y$ and buy $1.2$ units of $X$. Expected holding period is roughly 4--5 days based on the half-life. The position is exited when $s_t$ reverts toward zero.

---
## 7. Kalman Filter and State-Space Models

The Kalman filter is the optimal recursive estimator for linear Gaussian state-space models. It is used extensively in quantitative finance for estimating hidden states such as time-varying alpha, dynamic beta, and signal extraction.

### 7.1 Linear Gaussian State-Space Model

**State equation** (dynamics of the hidden state):

$$\boldsymbol{x}_t = \mathbf{F} \boldsymbol{x}_{t-1} + \mathbf{B} \boldsymbol{u}_t + \boldsymbol{w}_t, \qquad \boldsymbol{w}_t \sim N(\mathbf{0}, \mathbf{Q})$$

**Observation equation** (how we observe the hidden state):

$$\boldsymbol{y}_t = \mathbf{H} \boldsymbol{x}_t + \boldsymbol{v}_t, \qquad \boldsymbol{v}_t \sim N(\mathbf{0}, \mathbf{R})$$

where:
- $\boldsymbol{x}_t \in \mathbb{R}^n$: hidden state vector
- $\boldsymbol{y}_t \in \mathbb{R}^m$: observation vector
- $\mathbf{F}$: state transition matrix ($n \times n$)
- $\mathbf{H}$: observation matrix ($m \times n$)
- $\mathbf{Q}$: state noise covariance
- $\mathbf{R}$: observation noise covariance
- $\boldsymbol{w}_t$ and $\boldsymbol{v}_t$ are mutually independent

### 7.2 Kalman Filter Recursion

**Initialization:** $\hat{\boldsymbol{x}}_{0|0}$ and $\mathbf{P}_{0|0}$ (prior mean and covariance of state).

**Predict step** (time update):

\begin{align*}
\hat{\boldsymbol{x}}_{t|t-1} &= \mathbf{F} \hat{\boldsymbol{x}}_{t-1|t-1} + \mathbf{B}\boldsymbol{u}_t \\
\mathbf{P}_{t|t-1} &= \mathbf{F} \mathbf{P}_{t-1|t-1} \mathbf{F}^\top + \mathbf{Q}
\end{align*}

**Update step** (measurement update):

\begin{align*}
\boldsymbol{\nu}_t &= \boldsymbol{y}_t - \mathbf{H} \hat{\boldsymbol{x}}_{t|t-1} \qquad &\text{(innovation)} \\
\mathbf{S}_t &= \mathbf{H} \mathbf{P}_{t|t-1} \mathbf{H}^\top + \mathbf{R} \qquad &\text{(innovation covariance)} \\
\mathbf{K}_t &= \mathbf{P}_{t|t-1} \mathbf{H}^\top \mathbf{S}_t^{-1} \qquad &\text{(Kalman gain)} \\
\hat{\boldsymbol{x}}_{t|t} &= \hat{\boldsymbol{x}}_{t|t-1} + \mathbf{K}_t \boldsymbol{\nu}_t \qquad &\text{(updated state)} \\
\mathbf{P}_{t|t} &= (\mathbf{I} - \mathbf{K}_t \mathbf{H}) \mathbf{P}_{t|t-1} \qquad &\text{(updated covariance)}
\end{align*}

**Optimality:** Under the linear Gaussian assumption, the Kalman filter gives the exact conditional distribution $\boldsymbol{x}_t | \boldsymbol{y}_{1:t} \sim N(\hat{\boldsymbol{x}}_{t|t}, \mathbf{P}_{t|t})$. It minimizes the mean squared estimation error.

**Intuition for the Kalman gain:** $\mathbf{K}_t$ balances trust between the model prediction and the new observation. If observation noise $\mathbf{R}$ is large relative to state uncertainty $\mathbf{P}_{t|t-1}$, the gain is small (trust the model). If $\mathbf{R}$ is small, the gain is large (trust the data).

### 7.3 Finance Applications of the Kalman Filter

**Application 1: Time-Varying Beta.**

Model returns as $r_t = \alpha_t + \beta_t r_{m,t} + \varepsilon_t$ where $\beta_t$ follows a random walk:

\begin{align*}
\text{State:} \quad \begin{pmatrix} \alpha_t \\ \beta_t \end{pmatrix} &= \begin{pmatrix} \alpha_{t-1} \\ \beta_{t-1} \end{pmatrix} + \boldsymbol{w}_t \\
\text{Observation:} \quad r_t &= \begin{pmatrix} 1 & r_{m,t} \end{pmatrix} \begin{pmatrix} \alpha_t \\ \beta_t \end{pmatrix} + v_t
\end{align*}

The Kalman filter produces a smooth, real-time estimate of $\beta_t$ that adapts to regime changes.

**Application 2: Estimating Hidden Alpha.**

If a fund's alpha decays over time: $\alpha_t = \phi \alpha_{t-1} + w_t$ with $|\phi| < 1$, the Kalman filter estimates the current alpha from noisy return observations, distinguishing genuine skill from noise.

**Application 3: Pairs Trading Spread Dynamics.**

Model the cointegrating spread as a mean-reverting state with noisy observations. The Kalman filter provides optimal estimates of the current spread and its rate of reversion.

### 7.4 Relationship to Other Frameworks

Many time series models can be written in state-space form:

- **ARMA(p,q):** Cast as a state-space model with state dimension $\max(p,q+1)$.
- **Exponential smoothing:** Equivalent to a specific state-space model with particular $\mathbf{F}$, $\mathbf{H}$, $\mathbf{Q}$, $\mathbf{R}$.
- **Dynamic factor models:** Multiple observed series driven by a few latent factors.

The Kalman smoother extends the filter by using the full sample (past and future observations) to estimate each state, giving $\hat{\boldsymbol{x}}_{t|T}$ which has lower MSE than the filtered estimate $\hat{\boldsymbol{x}}_{t|t}$.

> 💡 **Interview Tip:** Be prepared to write out the Kalman filter recursion from memory. A common interview question is: "Explain intuitively what the Kalman gain does." The answer: it optimally weights new information (the innovation) against the prior prediction. Large gain means high trust in observations relative to the model; small gain means trust the model more. As more data arrives, the state estimate improves and the covariance $\mathbf{P}_{t|t}$ typically shrinks.

---
## 8. Spectral Analysis

Spectral analysis studies time series in the **frequency domain**, decomposing a signal into its constituent periodic components. This provides a complementary perspective to the time-domain methods above.

### 8.1 Spectral Density and the Spectral Representation

For a weakly stationary process with absolutely summable autocovariance function, the **spectral density** (or power spectral density) is the Fourier transform of the ACVF:

$$f(\omega) = \frac{1}{2\pi} \sum_{h=-\infty}^{\infty} \gamma(h) e^{-i\omega h}, \qquad \omega \in [-\pi, \pi]$$

The inverse relation recovers the autocovariance:

$$\gamma(h) = \int_{-\pi}^{\pi} f(\omega) e^{i\omega h} \, d\omega$$

In particular, $\gamma(0) = \text{Var}(X_t) = \int_{-\pi}^{\pi} f(\omega)\, d\omega$, so the spectral density decomposes the total variance across frequencies.

**Properties:**
- $f(\omega) \geq 0$ for all $\omega$ (positive semi-definiteness of $\gamma$)
- $f(\omega) = f(-\omega)$ (symmetry, since $\gamma(h) = \gamma(-h)$)
- Peaked at low frequencies $\Rightarrow$ persistent, slowly-varying behavior
- Peaked at high frequencies $\Rightarrow$ rapidly oscillating behavior
- White noise has a flat spectrum: $f(\omega) = \sigma^2 / (2\pi)$

### 8.2 Spectral Densities of Common Models

**AR(1) with** $X_t = \phi X_{t-1} + \varepsilon_t$:

$$f(\omega) = \frac{\sigma^2}{2\pi} \cdot \frac{1}{|1 - \phi e^{-i\omega}|^2} = \frac{\sigma^2}{2\pi(1 - 2\phi\cos\omega + \phi^2)}$$

- If $\phi > 0$: peaked at $\omega = 0$ (low-frequency dominance, persistence).
- If $\phi < 0$: peaked at $\omega = \pi$ (high-frequency dominance, oscillation).

**MA(1) with** $X_t = \varepsilon_t + \theta \varepsilon_{t-1}$:

$$f(\omega) = \frac{\sigma^2}{2\pi} |1 + \theta e^{-i\omega}|^2 = \frac{\sigma^2}{2\pi}(1 + 2\theta\cos\omega + \theta^2)$$

**General ARMA(p,q):**

$$f(\omega) = \frac{\sigma^2}{2\pi} \cdot \frac{|\Theta(e^{-i\omega})|^2}{|\Phi(e^{-i\omega})|^2}$$

This elegant formula shows that AR poles create peaks (resonances) in the spectrum, while MA zeros create troughs.

### 8.3 Periodogram and Spectral Estimation

The **periodogram** is the sample analogue of the spectral density. Given observations $X_1, \ldots, X_n$, the periodogram is:

$$I(\omega_j) = \frac{1}{n} \left| \sum_{t=1}^{n} X_t e^{-i\omega_j t} \right|^2$$

evaluated at the Fourier frequencies $\omega_j = 2\pi j / n$ for $j = 0, 1, \ldots, \lfloor n/2 \rfloor$.

**Key issue:** The periodogram is an **inconsistent** estimator of $f(\omega)$. Its variance does not decrease with sample size. In practice, we smooth the periodogram using:

- **Bartlett / Welch method:** Average periodograms over overlapping segments.
- **Kernel smoothing:** $\hat{f}(\omega) = \sum_{|h| \leq M} w(h/M) \hat{\gamma}(h) e^{-i\omega h}$ with a lag window $w(\cdot)$ and bandwidth $M$.
- **Multitaper methods (Thomson):** Use multiple orthogonal tapers to reduce variance.

### 8.4 Applications in Finance

- **Cycle detection:** Identify dominant periodicities in returns or economic indicators.
- **Signal vs noise:** Distinguish genuine periodic components from noise in trading signals.
- **Microstructure noise:** High-frequency data often has spectral signatures of bid-ask bounce (spike at $\omega = \pi$).
- **Filtering:** Design frequency-domain filters to extract specific components (e.g., band-pass filter for business cycle frequencies).

---
## 9. Finance Applications and Interview Problems

### 9.1 Return Predictability and Mean Reversion

If log prices follow a random walk, returns are unpredictable. Deviations from a random walk imply either predictability or time-varying risk premia.

**Variance ratio test.** Under a random walk, the variance of $k$-period returns scales linearly:

$$VR(k) = \frac{\text{Var}(r_t + r_{t-1} + \cdots + r_{t-k+1})}{k \cdot \text{Var}(r_t)} = 1$$

- $VR(k) > 1$: momentum (positive autocorrelation)
- $VR(k) < 1$: mean reversion (negative autocorrelation)

The Lo-MacKinlay (1988) test statistic under the null $VR(k) = 1$ is:

$$z(k) = \frac{VR(k) - 1}{\sqrt{\frac{2(2k-1)(k-1)}{3kn}}} \xrightarrow{d} N(0,1)$$

### 9.2 Ornstein-Uhlenbeck Process Calibration

The continuous-time mean-reverting (OU) process is:

$$dX_t = \kappa(\mu - X_t)\,dt + \sigma\,dW_t$$

where $\kappa > 0$ is the speed of mean reversion, $\mu$ is the long-run mean, and $\sigma$ is the volatility.

**Exact discretization:** Over a time step $\Delta t$:

$$X_{t+\Delta t} = \mu + (X_t - \mu)e^{-\kappa \Delta t} + \sigma \sqrt{\frac{1 - e^{-2\kappa \Delta t}}{2\kappa}} \cdot \epsilon_t, \qquad \epsilon_t \sim N(0,1)$$

This is an AR(1) in discrete time with:

\begin{align*}
\phi &= e^{-\kappa \Delta t} \\
c &= \mu(1 - e^{-\kappa \Delta t}) \\
\sigma_{\varepsilon}^2 &= \frac{\sigma^2}{2\kappa}(1 - e^{-2\kappa \Delta t})
\end{align*}

**Calibration procedure:**
1. Estimate the discrete AR(1) parameters $(c, \phi, \sigma_\varepsilon^2)$ from data.
2. Invert to get OU parameters:

\begin{align*}
\kappa &= -\frac{\ln \phi}{\Delta t} \\
\mu &= \frac{c}{1 - \phi} \\
\sigma &= \sigma_\varepsilon \sqrt{\frac{2\kappa}{1 - \phi^2}}
\end{align*}

**Half-life of the OU process:**

$$h_{1/2} = \frac{\ln 2}{\kappa}$$

### 9.3 Volatility Forecasting

Given a GARCH(1,1) model with parameters $(\omega, \alpha, \beta)$, the $h$-step ahead conditional variance forecast is:

$$\mathbb{E}[\sigma_{t+h}^2 | \mathcal{F}_t] = \bar{\sigma}^2 + (\alpha + \beta)^{h-1}(\sigma_{t+1}^2 - \bar{\sigma}^2)$$

where $\bar{\sigma}^2 = \omega/(1-\alpha-\beta)$ is the unconditional variance.

As $h \to \infty$, the forecast converges to $\bar{\sigma}^2$ (mean reversion of volatility).

The **term structure of volatility** (expected average variance over a horizon $H$) is:

$$\frac{1}{H}\sum_{h=1}^{H} \mathbb{E}[\sigma_{t+h}^2 | \mathcal{F}_t] = \bar{\sigma}^2 + \frac{(\sigma_{t+1}^2 - \bar{\sigma}^2)}{H} \cdot \frac{1 - (\alpha+\beta)^H}{1 - (\alpha+\beta)}$$

This is useful for pricing variance swaps and comparing with the VIX.

### Worked Problem 6: OU Calibration from Market Data

> **Problem.** You observe daily data for a spread $Z_t$ and estimate the AR(1) model $Z_t = 0.05 + 0.97 Z_{t-1} + \varepsilon_t$ with $\hat{\sigma}_\varepsilon = 0.12$. Calibrate the corresponding OU process $(\kappa, \mu, \sigma)$ with $\Delta t = 1/252$.

**Solution.**

*Speed of mean reversion:*

$$\kappa = -\frac{\ln \phi}{\Delta t} = -\frac{\ln 0.97}{1/252} = -252 \ln 0.97 = 252 \times 0.03046 = 7.675 \text{ per year}$$

*Long-run mean:*

$$\mu = \frac{c}{1-\phi} = \frac{0.05}{1-0.97} = \frac{0.05}{0.03} \approx 1.667$$

*Volatility:*

$$\sigma = \sigma_\varepsilon \sqrt{\frac{2\kappa}{1-\phi^2}} = 0.12 \sqrt{\frac{2 \times 7.675}{1-0.97^2}} = 0.12 \sqrt{\frac{15.35}{0.0591}} = 0.12 \sqrt{259.73} = 0.12 \times 16.12 \approx 1.934$$

*Half-life:*

$$h_{1/2} = \frac{\ln 2}{\kappa} = \frac{0.6931}{7.675} \approx 0.0903 \text{ years} \approx 22.7 \text{ trading days}$$

So the spread mean-reverts with a half-life of about 23 trading days, which is reasonable for a pairs trading strategy.

### Worked Problem 7: Spectral Density Derivation

> **Problem.** Derive the spectral density of the AR(1) process $X_t = \phi X_{t-1} + \varepsilon_t$, $\varepsilon_t \sim \text{WN}(0, \sigma^2)$, $|\phi| < 1$.

**Solution.**

The ACVF is $\gamma(h) = \frac{\sigma^2}{1-\phi^2} \phi^{|h|}$. The spectral density is:

\begin{align*}
f(\omega) &= \frac{1}{2\pi} \sum_{h=-\infty}^{\infty} \gamma(h) e^{-i\omega h} \\
&= \frac{\sigma^2}{2\pi(1-\phi^2)} \sum_{h=-\infty}^{\infty} \phi^{|h|} e^{-i\omega h}
\end{align*}

Split the sum:

\begin{align*}
\sum_{h=-\infty}^{\infty} \phi^{|h|} e^{-i\omega h} &= \sum_{h=0}^{\infty} (\phi e^{-i\omega})^h + \sum_{h=1}^{\infty} (\phi e^{i\omega})^h \\
&= \frac{1}{1-\phi e^{-i\omega}} + \frac{\phi e^{i\omega}}{1-\phi e^{i\omega}} \\
&= \frac{1 - \phi e^{i\omega} + \phi e^{i\omega} - \phi^2}{(1-\phi e^{-i\omega})(1-\phi e^{i\omega})} \\
&= \frac{1-\phi^2}{|1 - \phi e^{-i\omega}|^2}
\end{align*}

where $|1 - \phi e^{-i\omega}|^2 = (1-\phi e^{-i\omega})(1-\phi e^{i\omega}) = 1 - 2\phi\cos\omega + \phi^2$.

Therefore:

$$\boxed{f(\omega) = \frac{\sigma^2}{2\pi} \cdot \frac{1}{1 - 2\phi\cos\omega + \phi^2}}$$

### Worked Problem 8: Conditional Variance of AR(1)

> **Problem.** For the AR(1) model $X_t = \phi X_{t-1} + \varepsilon_t$ with $|\phi| < 1$ and $\varepsilon_t \sim \text{WN}(0, \sigma^2)$, derive the conditional variance $\text{Var}(X_{t+h} | X_t)$ and show it converges to the unconditional variance as $h \to \infty$.

**Solution.**

By forward iteration:

$$X_{t+h} = \phi^h X_t + \sum_{j=0}^{h-1} \phi^j \varepsilon_{t+h-j}$$

Since $X_t$ is known (conditioned on), and the innovations $\varepsilon_{t+1}, \ldots, \varepsilon_{t+h}$ are independent of $X_t$:

$$\text{Var}(X_{t+h} | X_t) = \text{Var}\left(\sum_{j=0}^{h-1} \phi^j \varepsilon_{t+h-j}\right) = \sigma^2 \sum_{j=0}^{h-1} \phi^{2j} = \sigma^2 \cdot \frac{1 - \phi^{2h}}{1 - \phi^2}$$

As $h \to \infty$:

$$\text{Var}(X_{t+h} | X_t) \to \frac{\sigma^2}{1-\phi^2} = \gamma(0) = \text{Var}(X_t) \qquad \checkmark$$

This confirms that uncertainty grows from zero (at $h=0$) to the unconditional variance (as $h \to \infty$), reflecting the loss of predictability over long horizons.

### Worked Problem 9: GARCH and the Leverage Effect

> **Problem.** Consider the GJR-GARCH model: $\sigma_t^2 = \omega + (\alpha + \gamma \mathbb{1}_{\{\varepsilon_{t-1}<0\}})\varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$ with $\omega = 0.00001$, $\alpha = 0.04$, $\gamma = 0.06$, $\beta = 0.90$, and current $\sigma_t^2 = 0.0004$. Compare the effect on tomorrow's variance of a $+2\%$ shock versus a $-2\%$ shock.

**Solution.**

*Positive shock:* $\varepsilon_t = +0.02$, so $\mathbb{1}_{\{\varepsilon_t < 0\}} = 0$.

\begin{align*}
\sigma_{t+1}^2 &= 0.00001 + 0.04 \times (0.02)^2 + 0.90 \times 0.0004 \\
&= 0.00001 + 0.04 \times 0.0004 + 0.00036 \\
&= 0.00001 + 0.000016 + 0.00036 = 0.000386
\end{align*}

*Negative shock:* $\varepsilon_t = -0.02$, so $\mathbb{1}_{\{\varepsilon_t < 0\}} = 1$.

\begin{align*}
\sigma_{t+1}^2 &= 0.00001 + (0.04 + 0.06) \times (0.02)^2 + 0.90 \times 0.0004 \\
&= 0.00001 + 0.10 \times 0.0004 + 0.00036 \\
&= 0.00001 + 0.00004 + 0.00036 = 0.000410
\end{align*}

The negative shock yields $\sigma_{t+1} \approx 2.025\%$ versus $\sigma_{t+1} \approx 1.965\%$ for the positive shock. The asymmetric response is:

$$\frac{0.000410 - 0.000386}{0.000386} \approx 6.2\%$$

The negative shock increases next-period variance by about 6.2% more than the positive shock of the same magnitude. This is the leverage effect: bad news has a disproportionate impact on volatility.

---
## Summary: Key Signatures for Model Identification

| Feature | AR(p) | MA(q) | ARMA(p,q) | Random Walk |
|---------|-------|-------|-----------|-------------|
| ACF | Decays (exp/osc) | Cuts off at lag $q$ | Decays after lag $q$ | Slow linear decay |
| PACF | Cuts off at lag $p$ | Decays (exp/osc) | Decays after lag $p$ | Spike at lag 1 only |
| Stationarity cond. | Roots of $\Phi(z)$ outside unit circle | Always stationary | Roots of $\Phi(z)$ outside unit circle | Non-stationary |
| Invertibility | Always invertible | Roots of $\Theta(z)$ outside unit circle | Roots of $\Theta(z)$ outside unit circle | N/A |

## Quick Reference: Key Formulas

**AR(1) moments:** $\mu = c/(1-\phi)$, $\gamma(0) = \sigma^2/(1-\phi^2)$, $\rho(h) = \phi^{|h|}$

**MA(1) ACF:** $\rho(1) = \theta/(1+\theta^2)$, $\rho(h) = 0$ for $|h| \geq 2$

**GARCH(1,1) long-run variance:** $\bar{\sigma}^2 = \omega/(1-\alpha-\beta)$

**OU half-life:** $h_{1/2} = \ln 2 / \kappa$

**AR(1) half-life:** $h_{1/2} = -\ln 2 / \ln|\phi|$

**GARCH volatility half-life:** $h_{1/2} = -\ln 2 / \ln(\alpha + \beta)$

**Kalman gain:** $\mathbf{K}_t = \mathbf{P}_{t|t-1}\mathbf{H}^\top(\mathbf{H}\mathbf{P}_{t|t-1}\mathbf{H}^\top + \mathbf{R})^{-1}$

**AR(1) spectral density:** $f(\omega) = \frac{\sigma^2}{2\pi(1 - 2\phi\cos\omega + \phi^2)}$

**Variance ratio:** $VR(k) = 1 + 2\sum_{j=1}^{k-1}(1 - j/k)\rho(j)$

**AIC:** $-2\ell + 2k$ &nbsp;&nbsp; **BIC:** $-2\ell + k\ln n$

## Interview Problem Set

Test your understanding with these additional practice questions.

**Q1.** Show that the unconditional variance of an AR(1) process $X_t = \phi X_{t-1} + \varepsilon_t$ blows up as $|\phi| \to 1$. What does this mean economically for a near-unit-root price process?

**Q2.** You have two models for daily returns: ARMA(1,1) with AIC = -5200, BIC = -5180, and ARMA(2,1) with AIC = -5210, BIC = -5175. Which model do you choose for (a) forecasting and (b) understanding the true data generating process?

**Q3.** A quant shows you a pairs trading backtest with a Sharpe ratio of 3.0. The spread half-life is 2 days. What questions would you ask to stress-test this result?

**Q4.** Explain why GARCH models produce heavier tails than the Gaussian distribution even when the innovations $z_t$ are Gaussian. Derive the excess kurtosis for ARCH(1).

**Q5.** In the Kalman filter, what happens to the Kalman gain as $\mathbf{R} \to 0$? As $\mathbf{R} \to \infty$? What is the practical interpretation?

### Solution Sketches

**A1.** $\text{Var}(X_t) = \sigma^2/(1-\phi^2)$. As $|\phi| \to 1$, $1 - \phi^2 \to 0$ and the variance diverges. Economically, a near-unit-root price series has enormous uncertainty about future levels, consistent with the efficient market hypothesis (prices are close to random walks).

**A2.** (a) For forecasting, AIC is preferred. ARMA(2,1) has lower AIC (-5210 vs -5200), so choose it. (b) For identifying the true model, BIC is preferred (consistent). ARMA(1,1) has lower BIC (-5180 vs -5175), so choose it. The extra AR term in ARMA(2,1) may be overfitting.

**A3.** Key questions: Is transaction cost included (2-day half-life means rapid turnover)? Is the cointegration stable out-of-sample? Is the backtest in-sample or walk-forward? How was the pair selected (data snooping bias)? What is the capacity (market impact on a 2-day horizon)? What happens during stress periods (cointegration breakdown)?

**A4.** The unconditional distribution of $\varepsilon_t$ is a mixture of Gaussians (conditional on $\sigma_t^2$, which varies). By the law of total variance and iterated expectations, this mixture has heavier tails. For ARCH(1): $\kappa = 3(1-\alpha^2)/(1-3\alpha^2) > 3$ when $\alpha > 0$.

**A5.** As $\mathbf{R} \to 0$ (perfect observations), $\mathbf{K}_t \to \mathbf{H}^{-1}$ and the state estimate jumps to match the observation exactly. As $\mathbf{R} \to \infty$ (infinitely noisy observations), $\mathbf{K}_t \to 0$ and the filter ignores the observation entirely, relying only on the state prediction.

---

*This notebook covers the essential time series theory for quantitative finance interviews. For computational practice, implement these models in Python using `statsmodels`, `arch`, and `filterpy`. Focus on building intuition for the key parameters (persistence, half-life, mean reversion speed) and their economic interpretation.*